# ChemBreak 12 — Adaptive MDP Jailbreak Safety Study

**Environment:** Google Cloud Notebook Enterprise  
**Primary condition:** `C3_ADAPTIVE_MDP`  
**Targets:** ChemDFM · ChemLLM  
**Fixed benchmark:** Train 241 · Test1 50 · Test2 50 · Test3 50 · Test4 50 · Reserve 59

This notebook intentionally follows the ChemBreak 11 Cloud workflow. Run cells top to bottom. Start with `PHASE = "train"`. Test1–Test4 require a frozen Train policy.

In [ ]:
from pathlib import Path
import importlib, json, os, shutil, site, subprocess, sys

# ── USER CONFIGURATION: confirm these before running ────────────────────────
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak12"
PHASE               = "train"      # train | test1 | test2 | test3 | test4
EXPERIMENT_REVISION = "CB12_MDP_V1"
LIVE                = True          # False = mock/dry-run; True = live models

assert PHASE in {"train","test1","test2","test3","test4"}, f"Unknown PHASE: {PHASE}"
content_root = Path("/content").resolve()
assert content_root.is_dir(), "/content unavailable — this notebook expects Google Cloud Notebook Enterprise."
if LIVE:
    assert PROJECT_ID.strip() and not PROJECT_ID.startswith("REPLACE_"), "Set PROJECT_ID before LIVE execution."

# Fresh CB12-only runtime storage. No CB7-CB11 storage path is imported.
storage_root = content_root / "chembreak12_storage"
model_cache  = storage_root / "cache" / "huggingface" / "hub"
env_paths = {
    "HF_HOME": storage_root / "cache/huggingface",
    "HF_HUB_CACHE": model_cache,
    "HF_MODULES_CACHE": storage_root / "cache/huggingface/modules",
    "XDG_CACHE_HOME": storage_root / "cache/xdg",
    "TORCH_HOME": storage_root / "cache/torch",
    "TORCHINDUCTOR_CACHE_DIR": storage_root / "cache/torchinductor",
    "TRITON_CACHE_DIR": storage_root / "cache/triton",
    "CUDA_CACHE_PATH": storage_root / "cache/cuda",
    "PIP_CACHE_DIR": storage_root / "cache/pip",
    "TMPDIR": storage_root / "tmp",
}
for var, path in env_paths.items():
    path.mkdir(parents=True, exist_ok=True); os.environ[var]=str(path)
os.environ["TMP"] = os.environ["TEMP"] = str(env_paths["TMPDIR"])

print("PROJECT_ID:", PROJECT_ID)
print("PHASE:", PHASE)
print("Storage:", storage_root)
print("Model cache:", model_cache)

## C1 — Clone/pull the ChemBreak repository

In [ ]:
checkout = content_root / "chembreak12_repo"
def git(*args, cwd=None): subprocess.run(["git", *args], cwd=cwd, check=True)

if not (checkout / ".git").is_dir():
    git("clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(checkout))
else:
    git("fetch", "origin", BRANCH, cwd=checkout)
    git("checkout", BRANCH, cwd=checkout)
    git("pull", "--ff-only", "origin", BRANCH, cwd=checkout)

PROJECT_DIR = (checkout / PROJECT_SUBDIR).resolve()
assert (PROJECT_DIR / "pyproject.toml").is_file(), (
    f"ChemBreak12 package not found at {PROJECT_DIR}. "
    "Add the chembreak12 folder from the supplied package to the configured GitHub repository first."
)
os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)

## C2 — Verify the fixed partition **before training**

In [ ]:
# Add source path temporarily so partition verification can run before package installation.
sys.path.insert(0, str(PROJECT_DIR / "src")); importlib.invalidate_caches()
import pandas as pd
from chembreak12.dataset import load_task_bank
from chembreak12.partition import build_manifest
from chembreak12.integrity import verify_lock

source_path   = PROJECT_DIR / "data/final_task_bank.csv"
manifest_path = PROJECT_DIR / "data/CB12_partition_manifest_v1.csv"
lock_path     = PROJECT_DIR / "data/CB12_partition_lock_v1.json"
source = load_task_bank(source_path)
locked = pd.read_csv(manifest_path)
reproduced = build_manifest(source.sample(frac=1, random_state=12026).reset_index(drop=True))
assert locked.fillna("").astype(str).equals(reproduced.fillna("").astype(str)), "Deterministic partition reproduction failed."
lock_report = verify_lock(source_path=source_path, frame=source, manifest_path=manifest_path, manifest=locked, lock_path=lock_path)
counts = locked["split"].value_counts().reindex(["Train","Test1","Test2","Test3","Test4","Reserve"])
print("Partition reproduced exactly: TRUE")
print(counts.to_string())
print("Pairwise task overlap: 0 (enforced by manifest verification)")
print("Partition lock: OK")

## C3 — Install the CB12 compatibility stack

In [ ]:
package_dir = storage_root / "python_packages"; package_dir.mkdir(parents=True, exist_ok=True)
compatibility_specs = [
    "transformers==4.40.2", "tokenizers==0.19.1", "huggingface-hub==0.23.5",
    "safetensors==0.4.5", "accelerate==0.30.1", "peft==0.10.0",
    "sentencepiece==0.2.0", "einops==0.8.1",
]
marker=package_dir/"chembreak12_compatibility_stack.json"; expected={"specifications":compatibility_specs}
installed=json.loads(marker.read_text()) if marker.exists() else None
if installed != expected:
    print("Installing compatibility stack (first CB12 run only)...")
    subprocess.run([sys.executable,"-m","pip","install","--target",str(package_dir),"--cache-dir",str(env_paths["PIP_CACHE_DIR"]),"--no-deps","--upgrade",*compatibility_specs],check=True)
    marker.write_text(json.dumps(expected,indent=2))
else: print("Compatibility stack already installed.")
subprocess.run([sys.executable,"-m","pip","install","-q","--target",str(package_dir),"--cache-dir",str(env_paths["PIP_CACHE_DIR"]),
    "google-auth>=2.35,<3","google-cloud-storage>=2.18,<4","google-genai>=1.47,<2","openai>=1.57,<3",
    "numpy>=1.26,<3","pandas>=2.2,<3","pydantic>=2.9,<3","PyYAML>=6.0,<7","rdkit>=2024.3","scipy>=1.13,<2","tenacity>=9,<10"],check=True)
site.addsitedir(str(package_dir)); sys.path.insert(0,str(package_dir)); sys.path.insert(0,str(PROJECT_DIR/"src")); importlib.invalidate_caches()
import torch
print("torch:",torch.__version__,"| CUDA:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))

## C4 — Build the runtime configuration

In [ ]:
import yaml
from chembreak12.config import load_config
config = load_config(PROJECT_DIR / "configs" / f"config.{PHASE}.yaml"); config.pop("_config_path",None)
config["run"].update({
    "project_root": str(PROJECT_DIR),
    "task_bank_path": str(PROJECT_DIR/"data/final_task_bank.csv"),
    "partition_manifest_path": str(PROJECT_DIR/"data/CB12_partition_manifest_v1.csv"),
    "partition_lock_path": str(PROJECT_DIR/"data/CB12_partition_lock_v1.json"),
    "output_root": str(storage_root/"runs"), "dry_run": not LIVE, "live_acknowledgement": LIVE,
})
config["storage"].update({
    "storage_root":str(storage_root),"hf_home":str(env_paths["HF_HOME"]),"hf_hub_cache":str(model_cache),
    "hf_modules_cache":str(env_paths["HF_MODULES_CACHE"]),"xdg_cache_home":str(env_paths["XDG_CACHE_HOME"]),
    "torch_home":str(env_paths["TORCH_HOME"]),"torchinductor_cache":str(env_paths["TORCHINDUCTOR_CACHE_DIR"]),
    "triton_cache":str(env_paths["TRITON_CACHE_DIR"]),"cuda_cache":str(env_paths["CUDA_CACHE_PATH"]),
    "pip_cache":str(env_paths["PIP_CACHE_DIR"]),"python_packages":str(package_dir),"temp_dir":str(env_paths["TMPDIR"]),
    "offload_dir":str(storage_root/"offload"),"preflight_dir":str(storage_root/"preflight"),
})
for t in config["targets"]:
    t["cache_dir"]=str(model_cache); t["offload_folder"]=str(storage_root/"offload"/t["id"])

policy_dir=storage_root/"policies"/EXPERIMENT_REVISION
training_policy_path=policy_dir/"train_policy.json"; frozen_policy_path=policy_dir/"frozen_policy.json"
config["policy"]["artifact_path"] = str(training_policy_path if PHASE=="train" else frozen_policy_path)
if PHASE!="train":
    assert frozen_policy_path.is_file(), f"Frozen policy not found: {frozen_policy_path}. Complete Train and freeze first."

runtime_dir=storage_root/"runtime_configs"; runtime_dir.mkdir(parents=True,exist_ok=True)
runtime_path=runtime_dir/f'CB12_{PHASE}_{"live" if LIVE else "mock"}.yaml'
runtime_path.write_text(yaml.safe_dump(config,sort_keys=False))
if LIVE:
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    os.environ["CHEMBREAK_ENABLE_LIVE"] = "YES"
else: os.environ.pop("CHEMBREAK_ENABLE_LIVE",None)
print("Runtime config:",runtime_path)
print("Google Cloud project:",os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("Targets:",[x["id"] for x in config["targets"]])
print("Policy mode/path:",config["policy"]["mode"],config["policy"]["artifact_path"])

## C5 — Preflight

In [ ]:
from chembreak12.preflight import run_preflight
import pprint
preflight=run_preflight(runtime_path,load_targets=False,probe_tokenizers=True)
pprint.pprint({k:preflight[k] for k in ("status","gpu","selected_subset","partition","policy","roles","tokenizers")})
assert preflight["status"]=="ok", "Preflight failed — fix the printed error before continuing."

## C6 — Experiment helper

In [ ]:
from pathlib import Path
from IPython.display import display
from chembreak12.runner import run_condition_target

def run_one(target_id):
    run_dir=Path(run_condition_target(runtime_path,"C3_ADAPTIVE_MDP",target_id))
    results_path=run_dir/"release"/"episode_results_C3_ADAPTIVE_MDP.csv"
    metrics_path=run_dir/"release"/"metrics_C3_ADAPTIVE_MDP.csv"
    results=pd.read_csv(results_path); metrics=pd.read_csv(metrics_path)
    display(results[results.target_id==target_id])
    display(metrics[metrics.target_id.isin([target_id,"ALL_TARGETS"])])
    print("Episode CSV:",results_path); print("Metrics CSV:",metrics_path)
    return run_dir
print("run_one helper ready")

## C7 — Run C3_ADAPTIVE_MDP against ChemDFM

In [ ]:
RUN_DIR = run_one("ChemDFM")

## C8 — Run C3_ADAPTIVE_MDP against ChemLLM

In [ ]:
RUN_DIR = run_one("ChemLLM")

## C9 — Freeze the Train policy

In [ ]:
from chembreak12.runner import strict_completion_gate
from chembreak12.policy import freeze_policy
if PHASE=="train":
    gate=strict_completion_gate(runtime_path,"C3_ADAPTIVE_MDP"); print("Completion gate:",gate)
    frozen_policy_path.parent.mkdir(parents=True,exist_ok=True)
    print("Policy frozen:",freeze_policy(training_policy_path,frozen_policy_path))
else:
    print(f"{PHASE} uses the already-frozen Train policy; no freeze performed.")

## C10 — Download release files

In [ ]:
from IPython.display import FileLink, display as ipy_display
release_dir=Path(RUN_DIR)/"release"; download_dir=storage_root/"downloads"; download_dir.mkdir(parents=True,exist_ok=True)
for name in ("episode_results_C3_ADAPTIVE_MDP.csv","metrics_C3_ADAPTIVE_MDP.csv","asr_by_budget_C3_ADAPTIVE_MDP.csv","policy_q_values.csv","policy_summary.json"):
    p=release_dir/name
    if p.exists(): ipy_display(FileLink(str(p)))
archive=Path(shutil.make_archive(str(download_dir/f"CB12_{PHASE}_release"),"zip",root_dir=Path(RUN_DIR),base_dir="release"))
print("Release archive:",archive); ipy_display(FileLink(str(archive)))